# 008 Built-in Middleware Recipes

这是 LangChain 学习线的第八份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/middleware/built-in

学习目标：

1. 逐个认识 LangChain built-in middleware
2. 每个 middleware 都给出最小用法样例
3. 解释它解决的问题和适用场景
4. 对比本仓库 Harness runtime 的已有能力
5. 判断是否适合直接接入当前项目

---

## 1. 准备：导入全部学习对象

为了避免真实模型调用，这一讲使用 `FakeListChatModel` 作为示例模型。

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

In [1]:
from langchain.agents.middleware import (
    AgentMiddleware,
    ClearToolUsesEdit,
    ContextEditingMiddleware,
    FilesystemFileSearchMiddleware,
    HumanInTheLoopMiddleware,
    LLMToolEmulator,
    LLMToolSelectorMiddleware,
    ModelCallLimitMiddleware,
    ModelFallbackMiddleware,
    ModelRetryMiddleware,
    PIIMiddleware,
    ShellToolMiddleware,
    SummarizationMiddleware,
    TodoListMiddleware,
    ToolCallLimitMiddleware,
    ToolRetryMiddleware,
)
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.tools import tool


fake_model = FakeListChatModel(responses=["fake response"])


def show(name: str, middleware):
    print(f"{name}: {type(middleware).__name__}")
    print(middleware)
    print()


## 2. SummarizationMiddleware

用途：上下文太长时，把旧消息总结成更短的上下文。

适合：

- 长对话
- 多步 agent
- 需要保留任务背景但不能无限塞历史

本仓库对应概念：

- `_compact_history(...)`
- context budget governance
- completed_steps 压缩

In [2]:
summarization = SummarizationMiddleware(
    model=fake_model,
    trigger=("messages", 12),
    keep=("messages", 6),
)
show("summarization", summarization)

summarization: SummarizationMiddleware



## 3. HumanInTheLoopMiddleware

用途：工具执行前中断，等待人工确认。

适合：

- 写文件
- shell 命令
- 发邮件、下单、删除数据

本仓库对应概念：

- `ApprovalTicket`
- `approval_required` SSE 事件
- `stream_resume_approval(...)`

In [3]:
hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "shell": True,
        "write_file": True,
    },
    description_prefix="工具执行需要人工审批",
)
show("human_in_the_loop", hitl)

human_in_the_loop: HumanInTheLoopMiddleware



## 4. ModelCallLimitMiddleware

用途：限制一次运行中的模型调用次数。

适合：

- 防止 agent 无限思考
- 控制成本
- 给复杂任务设硬边界

本仓库对应概念：

- `max_steps`
- Query Loop 停止条件

In [4]:
model_limit = ModelCallLimitMiddleware(run_limit=4, exit_behavior="end")
show("model_call_limit", model_limit)

model_call_limit: ModelCallLimitMiddleware



## 5. ToolCallLimitMiddleware

用途：限制工具调用次数，可以限制所有工具，也可以限制某一个工具。

适合：

- 防止 agent 重复调用搜索工具
- 防止 shell 或文件工具过度执行
- 降低循环失控风险

本仓库对应概念：

- tool permission
- loop step guard
- ledger 中的 tool-result 计数

In [5]:
tool_limit_all = ToolCallLimitMiddleware(run_limit=8, exit_behavior="continue")
tool_limit_shell = ToolCallLimitMiddleware(tool_name="shell", run_limit=1, exit_behavior="continue")
show("tool_call_limit_all", tool_limit_all)
show("tool_call_limit_shell", tool_limit_shell)

tool_call_limit_all: ToolCallLimitMiddleware

tool_call_limit_shell: ToolCallLimitMiddleware



## 6. ModelRetryMiddleware

用途：模型调用失败时重试。

适合：

- 临时网络问题
- 短暂限流
- 偶发模型服务错误

注意：

- 不要无限重试
- 业务错误不应该靠 retry 掩盖
- retry 仍然要进 ledger

In [6]:
model_retry = ModelRetryMiddleware(max_retries=2, initial_delay=0.5, max_delay=3.0)
show("model_retry", model_retry)

model_retry: ModelRetryMiddleware



## 7. ToolRetryMiddleware

用途：工具调用失败时重试。

适合：

- 外部 HTTP API 偶发失败
- 搜索接口偶发超时
- 只读工具短暂失败

不适合：

- 非幂等写操作
- 支付、删除、发邮件这类动作
- 参数错误导致的稳定失败

In [7]:
tool_retry = ToolRetryMiddleware(max_retries=2, tools=["web.search"], initial_delay=0.5, max_delay=3.0)
show("tool_retry", tool_retry)

tool_retry: ToolRetryMiddleware



## 8. ModelFallbackMiddleware

用途：主模型失败时切换备用模型。

适合：

- 主模型不可用
- 主模型限流
- 不同任务用不同能力模型兜底

本仓库对应概念：

- `ModelClient` 隔离模型访问
- fallback_error
- 可扩展为多模型路由

In [8]:
fallback = ModelFallbackMiddleware(fake_model, fake_model)
show("model_fallback", fallback)

model_fallback: ModelFallbackMiddleware



## 9. PIIMiddleware

用途：处理敏感信息。

内置类型包括：

- email
- credit_card
- ip
- mac_address
- url

常见策略：

- block
- redact
- mask
- hash

In [9]:
pii_email = PIIMiddleware("email", strategy="redact", apply_to_input=True)
pii_credit_card = PIIMiddleware("credit_card", strategy="mask", apply_to_output=True)
show("pii_email", pii_email)
show("pii_credit_card", pii_credit_card)

pii_email: PIIMiddleware

pii_credit_card: PIIMiddleware



## 10. LLMToolSelectorMiddleware

用途：工具很多时，先筛出一小批相关工具给模型。

适合：

- 工具数量很多
- 每次任务只需要其中少数工具
- 想降低 tools schema 上下文成本

本仓库对应概念：

- `SkillRegistry.select_skills(...)`
- preferred_tools
- pre-plan routing

In [10]:
tool_selector = LLMToolSelectorMiddleware(model=fake_model, max_tools=4, always_include=["echo"])
show("llm_tool_selector", tool_selector)

llm_tool_selector: LLMToolSelectorMiddleware



## 11. LLMToolEmulator

用途：让不支持原生工具调用的模型模拟工具调用。

适合：

- 私有模型网关不支持 tool calling
- 老模型只能输出文本
- 教学场景里想统一 agent 写法

注意：

这会降低可靠性。能用原生 tool calling 时，优先用原生 tool calling。

In [11]:
@tool
def echo_text(text: str) -> str:
    """Echo text back."""
    return text


tool_emulator = LLMToolEmulator(tools=[echo_text], model=fake_model)
show("llm_tool_emulator", tool_emulator)

llm_tool_emulator: LLMToolEmulator



## 12. ContextEditingMiddleware + ClearToolUsesEdit

用途：编辑上下文，尤其是清理过长的工具调用记录。

适合：

- 工具输出很大
- 只需要保留最近几次工具结果
- 避免历史工具输入反复占用上下文

本仓库对应概念：

- context compact
- completed_steps 摘要
- subagent synthesis

In [12]:
context_editing = ContextEditingMiddleware(
    edits=[
        ClearToolUsesEdit(trigger=20, keep=3, clear_tool_inputs=True),
    ],
)
show("context_editing", context_editing)

context_editing: ContextEditingMiddleware



## 13. TodoListMiddleware

用途：给复杂任务增加可见任务清单。

适合：

- 多步骤任务
- 需要展示进度
- 任务会根据结果调整

本仓库对应概念：

- ledger
- plan
- synthesis next_action

注意：简单问题不要强行 todo 化，会浪费 token。

In [13]:
todo = TodoListMiddleware()
show("todo_list", todo)

todo_list: TodoListMiddleware



## 14. FilesystemFileSearchMiddleware

用途：给 agent 增加文件搜索能力。

适合：

- 代码库只读调查
- 文档搜索
- repo research

本仓库对应概念：

- `ResearchSubAgent`
- `allowed_paths`
- pre-plan repo path routing

注意：一定要限制 `root_path`，不要让 agent 任意搜索整个系统。

In [14]:
file_search = FilesystemFileSearchMiddleware(root_path=".", use_ripgrep=True, max_file_size_mb=2)
show("filesystem_file_search", file_search)

filesystem_file_search: FilesystemFileSearchMiddleware



## 15. FilesystemMiddleware

官方文档里还有一个来自 Deep Agents 的 `FilesystemMiddleware`。

它和 `FilesystemFileSearchMiddleware` 的区别：

- `FilesystemFileSearchMiddleware` 更像只读搜索工具，适合 repo research
- `FilesystemMiddleware` 会提供 `ls`、`read_file`、`write_file`、`edit_file` 这类文件系统工具，范围更大，风险也更高

官方示例结构大致是：

```python
from langchain.agents import create_agent
from deepagents.middleware.filesystem import FilesystemMiddleware

agent = create_agent(
    model="claude-sonnet-4-6",
    middleware=[
        FilesystemMiddleware(
            backend=None,
            system_prompt="Write to the filesystem when...",
            custom_tool_descriptions={
                "ls": "Use the ls tool when...",
                "read_file": "Use the read_file tool to...",
            },
        ),
    ],
)
```

当前仓库没有安装 `deepagents`，所以这里不写可执行单元。学习时先记住判断：只读搜索可以低风险接入；写文件能力必须接 approval、allowed_paths 和审计。

## 16. ShellToolMiddleware

用途：给 agent 增加 shell 执行能力。

适合：

- 本地开发辅助
- 只读命令，如 `ls`、`rg`
- 受控脚本执行

风险：

- 可能删除文件
- 可能访问敏感信息
- 可能执行网络或安装命令

在本仓库里，这类能力必须走 approval 和 allowed_paths。

In [15]:
# 这里只创建 middleware，不执行任何 shell 命令。
shell = ShellToolMiddleware(workspace_root=".", tool_name="shell")
show("shell_tool", shell)

shell_tool: ShellToolMiddleware



## 17. SubAgentMiddleware

官方文档里还有一个来自 Deep Agents 的 `SubAgentMiddleware`。

用途：让主 agent 通过一个 task tool 把问题交给子 agent。它的核心价值不是“并行”，而是隔离上下文，让主 agent 不必把某个深度调查过程的全部细节都塞进自己的上下文。

官方示例结构大致是：

```python
from langchain.tools import tool
from langchain.agents import create_agent
from deepagents.middleware.subagents import SubAgentMiddleware

@tool
def get_weather(city: str) -> str:
    """Get the weather in a city."""
    return f"The weather in {city} is sunny."

agent = create_agent(
    model="claude-sonnet-4-6",
    middleware=[
        SubAgentMiddleware(
            default_model="claude-sonnet-4-6",
            default_tools=[],
            subagents=[
                {
                    "name": "weather",
                    "description": "This subagent can get weather in cities.",
                    "system_prompt": "Use the get_weather tool to get the weather in a city.",
                    "tools": [get_weather],
                    "model": "gpt-5.4",
                    "middleware": [],
                }
            ],
        )
    ],
)
```

和本仓库 Harness 设计对比：

- LangChain / Deep Agents：把 subagent 包成 middleware 能力
- 本仓库：由 planner 输出 `delegate`，再由 Harness 系统路由到 `ResearchSubAgent` / `VerificationSubAgent` / `ImplementationSubAgent`

当前仓库没有安装 `deepagents`，所以这里也不写可执行单元。学习重点是：子 agent 应该有独立目标、独立工具边界、独立上下文和主流程 synthesis。

## 18. AgentMiddleware

`AgentMiddleware` 是自定义 middleware 的基类。

当 built-in middleware 不满足业务需求时，可以继承它写自己的控制逻辑。

本仓库如果接 LangChain，自定义 middleware 的典型用途是：

- 把 LangChain 事件转换成 Harness ledger
- 接入现有 ApprovalTicket
- 接入现有 ToolRegistry
- 注入 session_id / run_id
- 做团队审计字段

In [16]:
base = AgentMiddleware()
show("agent_middleware_base", base)

agent_middleware_base: AgentMiddleware



## 19. 组合使用示例

一个比较合理的教学组合：

```python
middleware = [
    ModelCallLimitMiddleware(run_limit=4),
    ToolCallLimitMiddleware(run_limit=6),
    ToolRetryMiddleware(max_retries=2, tools=["web.search"]),
    PIIMiddleware("email", strategy="redact", apply_to_input=True),
    HumanInTheLoopMiddleware(interrupt_on={"shell": True}),
]
```

但如果接入本仓库页面，还要把中断、审批、恢复和 SSE 对齐。

In [21]:
teaching_middleware = [
    ModelCallLimitMiddleware(run_limit=4),
    ToolCallLimitMiddleware(run_limit=6),
    ToolRetryMiddleware(max_retries=2, tools=["web.search"]),
    PIIMiddleware("email", strategy="redact", apply_to_input=True),
    HumanInTheLoopMiddleware(interrupt_on={"shell": True}),
]

for item in teaching_middleware:
    print(type(item).__name__)

ModelCallLimitMiddleware
ToolCallLimitMiddleware
ToolRetryMiddleware
PIIMiddleware
HumanInTheLoopMiddleware


## 20. 实际环境运行：低风险 Agent Demo

前面的代码主要是在学习每个 middleware 怎么创建。

这一节真正接入当前项目 `.env` 里的模型配置，跑一个低风险 agent。

这个 demo 只做三件事：

1. 用真实模型决定是否调用工具
2. 用一个只读工具统计 `notebooks/langchain` 下的 Notebook 数量
3. 用 middleware 观察实际控制效果

接入的 middleware：

- `PIIMiddleware`：把用户输入里的邮箱脱敏
- `ModelCallLimitMiddleware`：限制本次 agent 最多调用几次模型
- `ToolCallLimitMiddleware`：限制本次 agent 最多调用几次工具
- `ToolRetryMiddleware`：工具异常时最多重试一次

这里不接 `ShellToolMiddleware` 和文件写入能力，因为它们需要审批、路径限制和审计，不适合直接在教学 Notebook 里自动执行。

In [18]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


load_dotenv(Path("../../.env"))
load_dotenv(Path(".env"))


@tool
def count_langchain_notebooks() -> str:
    """Count LangChain study notebooks in the local repository."""
    files = sorted(Path("notebooks/langchain").glob("*.ipynb"))
    return "当前 notebooks/langchain 下有 " + str(len(files)) + " 个 ipynb 文件：" + ", ".join(
        item.name for item in files
    )


api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("未配置 OPENAI_API_KEY，跳过真实模型调用。")
else:
    real_model = ChatOpenAI(
        model=os.getenv("OPENAI_MODEL") or "gpt-5.4-mini",
        base_url=os.getenv("OPENAI_BASE_URL") or None,
        temperature=0,
    )

    real_agent = create_agent(
        model=real_model,
        tools=[count_langchain_notebooks],
        system_prompt="你是学习助手。需要仓库信息时必须调用工具，然后用中文简短回答。",
        middleware=[
            ModelCallLimitMiddleware(run_limit=4, exit_behavior="error"),
            ToolCallLimitMiddleware(run_limit=2, exit_behavior="error"),
            ToolRetryMiddleware(max_retries=1, tools=["count_langchain_notebooks"]),
            PIIMiddleware("email", strategy="redact", apply_to_input=True),
        ],
    )

    result = real_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "我的邮箱是 java.user@example.com。请检查当前 LangChain 学习 Notebook 有多少份？",
                }
            ]
        }
    )

    for index, message in enumerate(result.get("messages", [])):
        print("--- message", index, "type=", getattr(message, "type", type(message).__name__))
        print(getattr(message, "content", ""))
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)


--- message 0 type= human
我的邮箱是 [REDACTED_EMAIL]。请检查当前 LangChain 学习 Notebook 有多少份？
--- message 1 type= ai



tool_calls: [{'name': 'count_langchain_notebooks', 'args': {}, 'id': 'call_151a00aadb1144d2bde7d598', 'type': 'tool_call'}]
--- message 2 type= tool
当前 notebooks/langchain 下有 8 个 ipynb 文件：001-langchain-overview-and-agent.ipynb, 002-models-and-messages.ipynb, 003-tools-and-tool-calling.ipynb, 004-structured-output.ipynb, 005-agents-and-control-flow.ipynb, 006-langchain-in-fastapi.ipynb, 007-built-in-middleware.ipynb, 008-built-in-middleware-recipes.ipynb
--- message 3 type= ai


当前仓库中有 8 份 LangChain 学习 Notebook。


## 21. 本讲小结

这一讲逐个看了：

- SummarizationMiddleware
- HumanInTheLoopMiddleware
- ModelCallLimitMiddleware
- ToolCallLimitMiddleware
- ModelRetryMiddleware
- ToolRetryMiddleware
- ModelFallbackMiddleware
- PIIMiddleware
- LLMToolSelectorMiddleware
- LLMToolEmulator
- ContextEditingMiddleware
- ClearToolUsesEdit
- TodoListMiddleware
- FilesystemFileSearchMiddleware
- FilesystemMiddleware
- ShellToolMiddleware
- SubAgentMiddleware
- AgentMiddleware

最重要的判断：

```text
middleware 是控制面组件，但业务边界仍然要由业务系统定义。
```

下一步如果继续推进，可以开始做一个真正的 `/langchain-study` 教学接口，只接低风险 tool，并观察它和当前 Harness runtime 的差异。